In [1]:
# %%
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

import plotly.graph_objects as go

from collections import defaultdict
import random

np.random.seed(42)
torch.manual_seed(42)

In [2]:
# %%
def parse_array(x):
    if isinstance(x, list):
        return np.array(x, dtype=np.float32)
    
    if isinstance(x, str):
        x = x.strip("[]")
        x = x.replace(",", " ")
        return np.fromstring(x, sep=" ", dtype=np.float32)
    
    return np.zeros(0, dtype=np.float32)

In [3]:
# %%
def build_trajectories(sensor_df):
    sensor_df = sensor_df.sort_values(["trackId", "timestamp"])
    
    trajs = {}
    for tid, g in sensor_df.groupby("trackId"):
        trajs[tid] = g[[
            "latitude", "longitude", "heading", "speed"
        ]].values.astype(np.float32)
        
    return trajs

In [4]:
# %%
def sample_subtracks(trajs, L=12, stride=3):
    subs = []
    
    for tid, traj in trajs.items():
        if len(traj) < L:
            continue
        
        for i in range(0, len(traj) - L + 1, stride):
            subs.append({
                "track_id": tid,
                "seq": traj[i:i+L]
            })
    
    return subs

In [5]:
# %%
def encode_report_seq(row, L=12):
    lons = parse_array(row["coors.longitudes"])
    lats = parse_array(row["coors.latitudes"])
    
    n = min(len(lons), len(lats))
    if n < 2:
        return np.zeros((L, 2), dtype=np.float32)
    
    pts = np.stack([lats[:n], lons[:n]], axis=1)
    
    idx = np.linspace(0, n-1, L).astype(int)
    return pts[idx].astype(np.float32)

In [6]:
# %%
class TrajectoryEncoder(nn.Module):
    def __init__(self, in_dim, d_model=64):
        super().__init__()
        
        self.input_proj = nn.Linear(in_dim, d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )
        
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.pool = nn.AdaptiveAvgPool1d(1)
    
    def forward(self, x):
        x = self.input_proj(x)
        h = self.encoder(x)
        h = h.transpose(1,2)
        h = self.pool(h).squeeze(-1)
        return F.normalize(h, dim=1)

In [15]:
# %%
def dtw_distance(a, b):
    n, m = len(a), len(b)
    dp = np.full((n+1, m+1), np.inf)
    dp[0,0] = 0
    
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = np.linalg.norm(a[i-1][:2] - b[j-1][:2])
            dp[i,j] = cost + min(
                dp[i-1,j],
                dp[i,j-1],
                dp[i-1,j-1]
            )
    return dp[n,m]

In [8]:
# %%
def build_candidates(sub_tracks, report_seqs, radius=0.2, top_k=20):
    candidates = []
    
    for s in sub_tracks:
        start = s["seq"][0][:2]
        
        dists = []
        for rid, r in enumerate(report_seqs):
            d = np.linalg.norm(start - r[0])
            dists.append(d)
        
        dists = np.array(dists)
        
        idx = np.where(dists < radius)[0]
        if len(idx) == 0:
            idx = np.argsort(dists)[:top_k]
        
        candidates.append(idx)
    
    return candidates

In [9]:
# %%
def build_training_batches(sub_tracks, report_seqs, candidates, batch_size=32):
    
    while True:
        idx = np.random.choice(len(sub_tracks), batch_size)
        
        t_batch = []
        r_pos = []
        r_neg = []
        
        for i in idx:
            t_batch.append(sub_tracks[i]["seq"])
            
            cand = candidates[i]
            
            # positive = nearest
            pos = cand[0]
            neg = random.choice(range(len(report_seqs)))
            
            r_pos.append(report_seqs[pos])
            r_neg.append(report_seqs[neg])
        
        yield (
            torch.tensor(t_batch),
            torch.tensor(r_pos),
            torch.tensor(r_neg)
        )

In [10]:
# %%
def train_model(sub_tracks, report_seqs, candidates, epochs=5):
    
    net_t = TrajectoryEncoder(4)
    net_r = TrajectoryEncoder(2)
    
    opt = torch.optim.Adam(
        list(net_t.parameters()) + list(net_r.parameters()),
        lr=1e-3
    )
    
    loader = build_training_batches(sub_tracks, report_seqs, candidates)
    
    for ep in range(epochs):
        total = 0
        
        for _ in range(200):
            t, r_pos, r_neg = next(loader)
            
            e_t = net_t(t)
            e_pos = net_r(r_pos)
            e_neg = net_r(r_neg)
            
            pos_sim = torch.sum(e_t * e_pos, dim=1)
            neg_sim = torch.sum(e_t * e_neg, dim=1)
            
            loss = F.relu(1.0 - pos_sim + neg_sim).mean()
            
            opt.zero_grad()
            loss.backward()
            opt.step()
            
            total += loss.item()
        
        print(f"Epoch {ep+1}: loss={total:.4f}")
    
    return net_t, net_r

In [11]:
# %%
def match_reports(sub_tracks, report_seqs, net_t, net_r, top_k=5):
    
    results = {}
    
    with torch.no_grad():
        for rid, r_seq in enumerate(report_seqs):
            
            r_emb = net_r(torch.tensor(r_seq).unsqueeze(0)).squeeze()
            
            scores = []
            
            for i, s in enumerate(sub_tracks):
                t_seq = s["seq"]
                t_emb = net_t(torch.tensor(t_seq).unsqueeze(0)).squeeze()
                
                sim = torch.dot(t_emb, r_emb).item()
                dtw = dtw_distance(t_seq, r_seq)
                
                score = 0.6 * sim + 0.4 * np.exp(-dtw)
                
                scores.append((s["track_id"], score))
            
            scores.sort(key=lambda x: -x[1])
            results[rid] = scores[:top_k]
    
    return results

In [12]:
# %%
def compute_uncertainty(scores):
    vals = np.array([s for _, s in scores])
    probs = np.exp(vals) / np.sum(np.exp(vals))
    
    entropy = -np.sum(probs * np.log(probs + 1e-8))
    return entropy

In [13]:
# %%
def plot_matches(sensor_df, event_df, results, num_reports=5):
    
    fig = go.Figure()
    
    for rid in list(results.keys())[:num_reports]:
        row = event_df.iloc[rid]
        
        lons = parse_array(row["coors.longitudes"])
        lats = parse_array(row["coors.latitudes"])
        
        fig.add_trace(go.Scatter(
            x=lons, y=lats,
            mode="lines+markers",
            name=f"Report {rid}",
            line=dict(width=4)
        ))
        
        for rank, (tid, sc) in enumerate(results[rid]):
            track = sensor_df[sensor_df["trackId"] == tid]
            
            fig.add_trace(go.Scatter(
                x=track["longitude"],
                y=track["latitude"],
                mode="lines",
                name=f"T{tid} rank{rank+1} ({sc:.2f})"
            ))
    
    fig.update_layout(
        title="Matching (Offline)",
        xaxis_title="Longitude",
        yaxis_title="Latitude",
        height=700
    )
    
    fig.show()

In [16]:
# %%
sensor_df = pd.read_parquet("sensor.parquet")
event_df  = pd.read_parquet("event.parquet")

trajs = build_trajectories(sensor_df)
sub_tracks = sample_subtracks(trajs)

report_seqs = [
    encode_report_seq(row) for row in event_df.to_dict("records")
]

candidates = build_candidates(sub_tracks, report_seqs)

net_t, net_r = train_model(sub_tracks, report_seqs, candidates)

results = match_reports(sub_tracks, report_seqs, net_t, net_r)

plot_matches(sensor_df, event_df, results)

Epoch 1: loss=200.0322
Epoch 2: loss=199.9878
Epoch 3: loss=200.0622
Epoch 4: loss=199.8428
Epoch 5: loss=200.1522
